In [ ]:
import numpy as np
import scipy.signal as si
import matplotlib.pyplot as plt

from scipy.constants import c

In [ ]:
from lisaorbits import LINKS

## Exercise 1: Different orbit types

1. Explore the orbit types `EqualArmlengthOrbits`, `KeplerianOrbits` and `StaticConstellation` (use `orbits = StaticConstellation.from_orbits` to freeze either `EqualArmlengthOrbits` or `KeplerianOrbits` in time). Plot spacecraft positions and light travel times and compare them qualitatively.
2. Do `EqualArmlengthOrbits` actually produce equal arm length? What is the effect of setting `tt_order=0` and `ignore_shapiro=True` in the constructor. What is the meaning of those two parameters?

In [ ]:
from lisaorbits import StaticConstellation, EqualArmlengthOrbits, KeplerianOrbits

In [ ]:
orbits_equal = EqualArmlengthOrbits()
orbits_keplerian = KeplerianOrbits()
orbits_static = StaticConstellation.from_orbits(orbits_keplerian, 100000)

times = np.linspace(0, 3600 * 24 * 365, 1000) # in seconds

In [ ]:
ltts = orbits_equal.compute_ltt(times)

plt.plot(times / (3600 * 24 * 365), ltts, label=[f"Link {mosa}" for mosa in LINKS])

plt.xlabel("Times (days)")
plt.ylabel("Light travel times (s)")
plt.legend()

plt.figure()

ltts = orbits_keplerian.compute_ltt(times)

plt.plot(times / (3600 * 24 * 365), ltts, label=[f"Link {mosa}" for mosa in LINKS])

plt.xlabel("Times (days)")
plt.ylabel("Light travel times (s)")
plt.legend()

plt.figure()

ltts = orbits_static.compute_ltt(times)

plt.plot(times / (3600 * 24 * 365), ltts, label=[f"Link {mosa}" for mosa in LINKS])

plt.xlabel("Times (days)")
plt.ylabel("Light travel times (s)")
plt.legend()

## Exercise 2: Numerical orbits provided by ESA

The most realistic orbits include the gravitional effects of all relevant planets in the solar system (Mercury, Venus, Earth + Moon, Mars, Jupiter and Saturn). In order to stay within the mission constraints the initial injection of the spacecraft has to be optimized. ESA has provided orbit files for a 10-year science phase.

1. ESA orbit files can be accessed by calling `orbits = OEMOrbits.from_included("esa-trailing")`. Moreover, they can only be evaluated for times within `orbits.t_start` and `orbits.t_end`. Plot the light travel times between the spacecraft. What do you notice?
2. Check whether the orbit stability requirements are fulfilled:
    - The light travel time shall not vary by more than $10\%$.
    - The relative velocities (light travel time derivatives) shall not exceed $10 \,\text{m/s}$ ($3.3 \times 10^{-8} \,\text{s/s}$). Hint: Use the function `compute_ltt_derivative`.
    - The corner angles shall not exceed $60 \pm 1$ degree. Hint: Use the function `compute_unit_vector`

In [ ]:
from lisaorbits import OEMOrbits

orbits = OEMOrbits.from_included("esa-trailing")
times = np.linspace(orbits.t_start, orbits.t_end, 1000) # in seconds

In [ ]:
ltts = orbits.compute_ltt(times)

plt.plot(times / (3600 * 24 * 365), ltts, label=[f"Link {mosa}" for mosa in LINKS])

plt.axhline(np.mean(ltts) * 1.1, c="grey", ls="--")
plt.axhline(np.mean(ltts) * 0.9, c="grey", ls="--")

plt.xlabel("Times (days)")
plt.ylabel("Light travel times (s)")
plt.legend()

In [ ]:
ltt_derivatives = orbits.compute_ltt_derivative(times)

plt.plot(times / (3600 * 24 * 365), ltt_derivatives, label=[f"Link {mosa}" for mosa in LINKS])

plt.axhline(3.3e-8, c="grey", ls="--")
plt.axhline(-3.3e-8, c="grey", ls="--")

plt.xlabel("Times (days)")
plt.ylabel("Light travel time derivatives (s/s)")
plt.legend()

In [ ]:
unit_vectors = orbits.compute_unit_vector(times)

print("standard order:", LINKS)
print("shape of `unit_vectors` (times, links, components):", unit_vectors.shape)

# calculate dot product between unit vector and convert to an angle (a . b = cos(theta))
angles_1 = np.arccos(np.einsum("...i,...i", unit_vectors[:,0], unit_vectors[:,3]))
angles_2 = np.arccos(np.einsum("...i,...i", unit_vectors[:,1], unit_vectors[:,5]))
angles_3 = np.arccos(np.einsum("...i,...i", unit_vectors[:,2], unit_vectors[:,4]))

plt.plot(times / (3600 * 24 * 365), angles_1 * 180 / np.pi, label="SC 1")
plt.plot(times / (3600 * 24 * 365), angles_2 * 180 / np.pi, label="SC 2")
plt.plot(times / (3600 * 24 * 365), angles_3 * 180 / np.pi, label="SC 3")

plt.axhline(59, c="grey", ls="--")
plt.axhline(61, c="grey", ls="--")

plt.xlabel("Times (days)")
plt.ylabel("Corner angles (degrees)")
plt.legend()